# 02 - Bell States, GHZ States, and Deterministic Hashing

This notebook creates three entangled circuits of increasing size and shows how
QONTOS normalizes each one to a canonical `CircuitIR`. The key takeaway:
**`circuit_hash` is deterministic** -- the same logical circuit always produces
the same hash, enabling deduplication, caching, and integrity checks across
the entire pipeline.

In [ ]:
from qontos.circuit import CircuitNormalizer
from qontos.circuit.metadata import extract_metadata

normalizer = CircuitNormalizer()

## Circuit Definitions

We define three circuits:

| Circuit | Qubits | State | Description |
|---------|--------|-------|-------------|
| Bell    | 2      | $\|\Phi^+\rangle$ | Simplest entangled state |
| GHZ-3   | 3      | $\frac{1}{\sqrt{2}}(\|000\rangle + \|111\rangle)$ | Three-qubit GHZ |
| GHZ-5   | 5      | $\frac{1}{\sqrt{2}}(\|00000\rangle + \|11111\rangle)$ | Five-qubit GHZ |

In [ ]:
bell_qasm = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

ghz3_qasm = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];
h q[0];
cx q[0],q[1];
cx q[1],q[2];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

ghz5_qasm = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[5];
creg c[5];
h q[0];
cx q[0],q[1];
cx q[1],q[2];
cx q[2],q[3];
cx q[3],q[4];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
measure q[3] -> c[3];
measure q[4] -> c[4];
"""

circuits = {
    "Bell (2Q)": bell_qasm,
    "GHZ-3 (3Q)": ghz3_qasm,
    "GHZ-5 (5Q)": ghz5_qasm,
}

## Normalize All Circuits

Each QASM string goes through `CircuitNormalizer.normalize()` to produce a
`CircuitIR`. The normalizer parses the QASM, extracts the gate list,
computes connectivity, and generates a deterministic hash.

In [ ]:
normalized = {}

for name, qasm in circuits.items():
    ir = normalizer.normalize(input_type="openqasm", source=qasm)
    normalized[name] = ir
    print(f"{name}:")
    print(f"  Qubits     : {ir.num_qubits}")
    print(f"  Depth      : {ir.depth}")
    print(f"  Gate count : {ir.gate_count}")
    print(f"  Hash       : {ir.circuit_hash[:24]}...")
    print()

## Deterministic Hashing

The `circuit_hash` is a SHA-256 digest of the circuit's canonical JSON form.
This means:

- **Same circuit = same hash**, regardless of whitespace or formatting in the QASM
- **Different circuits = different hashes**
- Hashes can be used for deduplication, caching, and integrity proofs

Let's verify both properties.

In [ ]:
# Property 1: Same circuit always produces the same hash
ir_a = normalizer.normalize(input_type="openqasm", source=bell_qasm)
ir_b = normalizer.normalize(input_type="openqasm", source=bell_qasm)
assert ir_a.circuit_hash == ir_b.circuit_hash
print("[PASS] Same circuit -> same hash")

# Property 2: Different circuits produce different hashes
hashes = [ir.circuit_hash for ir in normalized.values()]
assert len(hashes) == len(set(hashes)), "Hash collision detected!"
print("[PASS] Different circuits -> different hashes")

print("\nHash summary:")
for name, ir in normalized.items():
    print(f"  {name:<14} -> {ir.circuit_hash[:32]}...")

## Metadata Comparison

As circuits grow, the metadata reveals how complexity scales. Notice that
entanglement density and connectivity edges increase with GHZ size.

In [ ]:
# Build a comparison table
headers = ["Metric", "Bell (2Q)", "GHZ-3 (3Q)", "GHZ-5 (5Q)"]
meta = {name: extract_metadata(ir) for name, ir in normalized.items()}

rows = [
    ("1Q gates",           [m["single_qubit_gates"] for m in meta.values()]),
    ("2Q gates",           [m["two_qubit_gates"] for m in meta.values()]),
    ("Entanglement",       [f"{m['entanglement_density']:.4f}" for m in meta.values()]),
    ("Parallelism",        [f"{m['parallelism_ratio']:.4f}" for m in meta.values()]),
    ("Connectivity edges", [m["connectivity_edges"] for m in meta.values()]),
    ("Gate types",         [m["unique_gate_types"] for m in meta.values()]),
    ("Has measurements",   [m["has_measurements"] for m in meta.values()]),
]

# Print as table
print(f"{'Metric':<24} {'Bell (2Q)':>12} {'GHZ-3 (3Q)':>12} {'GHZ-5 (5Q)':>12}")
print("-" * 60)
for label, values in rows:
    print(f"{label:<24} {str(values[0]):>12} {str(values[1]):>12} {str(values[2]):>12}")

## Connectivity Visualization

The `qubit_connectivity` field stores pairs of qubits connected by two-qubit gates.
For GHZ circuits this forms a linear chain, but more complex circuits can have
arbitrary topologies that influence partitioning decisions.

In [ ]:
for name, ir in normalized.items():
    print(f"{name}:")
    print(f"  Qubits: {list(range(ir.num_qubits))}")
    print(f"  Edges : {ir.qubit_connectivity}")
    
    # Simple ASCII connectivity diagram
    diagram = " -- ".join(f"q{i}" for i in range(ir.num_qubits))
    print(f"  Chain : {diagram}")
    print()

## Summary

In this notebook you learned:

- How to create Bell and GHZ circuits of varying sizes
- That `circuit_hash` is **deterministic**: same circuit always yields the same hash
- How metadata scales with circuit size (entanglement density, connectivity)
- How `qubit_connectivity` captures the circuit's topology

**Next steps:**
- [03_circuit_partitioning.ipynb](03_circuit_partitioning.ipynb) shows how QONTOS
  splits larger circuits into modules for distributed execution